In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/task-3-nlp-olympic-ai-neu/val.csv
/kaggle/input/task-3-nlp-olympic-ai-neu/train.csv
/kaggle/input/task-3-nlp-olympic-ai-neu/test.csv


# Vietnamese Question Answering (ViQA) - BiLSTM Implementation

## Task Overview

Implement a BiLSTM-based approach for Vietnamese Question Answering with the following simplified architecture:

1. Word Embedding Layer (trained from scratch)
2. BiLSTM Encoder for Review and Question
3. Simple similarity-based span extraction
4. Position prediction for start/end of answer spans

## Architecture Design

- **Embedding**: Word2Vec trained on corpus
- **Encoder**: Bidirectional LSTM for sequence encoding
- **Matching**: Cosine similarity between question and review segments
- **Output**: Start/end position prediction for answer spans

## Step 1: Import Required Libraries

Import necessary libraries for deep learning implementation using TensorFlow/Keras.

In [2]:
import pandas as pd
import numpy as np
import re
import string
from typing import List, Tuple, Dict
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Traditional NLP libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import difflib

# Lightweight Deep Learning for embeddings
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# Evaluation metrics
from sklearn.metrics import accuracy_score

print("All libraries imported successfully")
print(" baseline with hybrid TF-IDF + Word2Vec approach")

All libraries imported successfully
 baseline with hybrid TF-IDF + Word2Vec approach


## Step 2: Load and Explore Dataset

Load the datasets and examine the data structure for preprocessing.

In [3]:
# Load datasets
train_df = pd.read_csv('/kaggle/input/task-3-nlp-olympic-ai-neu/train.csv')
val_df = pd.read_csv('/kaggle/input/task-3-nlp-olympic-ai-neu/val.csv')
test_df = pd.read_csv('/kaggle/input/task-3-nlp-olympic-ai-neu/test.csv')

print("Dataset Overview:")
print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")

# Use subset for faster development and training
TRAIN_SUBSET_SIZE = 2000  # Increased from 1000 for better model
VAL_SUBSET_SIZE = 300     # Increased validation set

train_subset = train_df.head(TRAIN_SUBSET_SIZE)
val_subset = val_df.head(VAL_SUBSET_SIZE)

print(f"\nUsing subsets for efficient training:")
print(f"Train subset: {len(train_subset)} samples")
print(f"Val subset: {len(val_subset)} samples")

# Examine data structure
print(f"\nSample data structure:")
sample = train_df.iloc[0]
print(f"Columns: {train_df.columns.tolist()}")
print(f"Review (first 100 chars): {sample['review'][:100]}...")
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answer']}")

Dataset Overview:
Training set: 4154 samples
Validation set: 890 samples
Test set: 890 samples

Using subsets for efficient training:
Train subset: 2000 samples
Val subset: 300 samples

Sample data structure:
Columns: ['id', 'review', 'question', 'answer']
Review (first 100 chars): Nơi đây thích hợp cho việc hộp nhóm , đọc sách , nghe nhạc , thư giãn .
- Có nhiều sách để đọc : Tru...
Question: Quán phù hợp nhất cho những khách hàng có mục đích gì?
Answer: thích hợp cho việc hộp nhóm , đọc sách , nghe nhạc , thư giãn


## Step 3: Text Preprocessing and Tokenization

Implement text preprocessing specifically for deep learning models.

In [4]:
def clean_vietnamese_text(text):
    """ Vietnamese text cleaning for better processing"""
    if pd.isna(text):
        return ""
    
    # Convert to string and strip
    text = str(text).strip()
    
    # Remove extra whitespace and newlines
    text = re.sub(r'\s+', ' ', text)
    
    # Keep Vietnamese characters, basic punctuation, and numbers
    text = re.sub(r'[^\w\s\u00C0-\u1EF9.,!?0-9]', ' ', text)
    
    # Convert to lowercase for consistency
    text = text.lower()
    
    return text.strip()

def extract_ngrams(text, n=3):
    """Extract character n-grams for robust matching"""
    text = clean_vietnamese_text(text)
    if len(text) < n:
        return [text]
    return [text[i:i+n] for i in range(len(text)-n+1)]

def extract_word_ngrams(text, n=2):
    """Extract word n-grams for semantic matching"""
    words = clean_vietnamese_text(text).split()
    if len(words) < n:
        return [' '.join(words)]
    return [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]

# Test preprocessing
sample_text = "Sản phẩm này rất tốt và chất lượng cao!"
print("Preprocessing test:")
print(f"Original: {sample_text}")
print(f"Cleaned: {clean_vietnamese_text(sample_text)}")
print(f"Char 3-grams: {extract_ngrams(sample_text, 3)[:5]}...")  # Show first 5
print(f"Word 2-grams: {extract_word_ngrams(sample_text, 2)}")

print("\nText preprocessing functions ready!")

Preprocessing test:
Original: Sản phẩm này rất tốt và chất lượng cao!
Cleaned: sản phẩm này rất tốt và chất lượng cao!
Char 3-grams: ['sản', 'ản ', 'n p', ' ph', 'phẩ']...
Word 2-grams: ['sản phẩm', 'phẩm này', 'này rất', 'rất tốt', 'tốt và', 'và chất', 'chất lượng', 'lượng cao!']

Text preprocessing functions ready!


## Step 4: Build Word2Vec Embeddings

Train Word2Vec embeddings from scratch on the corpus.

In [5]:
# Prepare corpus for Word2Vec training
def prepare_corpus_for_embeddings(df):
    """Prepare text corpus for embedding training"""
    sentences = []
    
    for _, row in df.iterrows():
        # Add review sentences
        review_clean = clean_vietnamese_text(row['review'])
        if review_clean:
            sentences.append(review_clean.split())
        
        # Add question sentences
        question_clean = clean_vietnamese_text(row['question'])
        if question_clean:
            sentences.append(question_clean.split())
        
        # Add answer sentences (if available)
        if 'answer' in row and pd.notna(row['answer']):
            answer_clean = clean_vietnamese_text(row['answer'])
            if answer_clean:
                sentences.append(answer_clean.split())
    
    return sentences

# Build Word2Vec corpus
print("Preparing corpus for Word2Vec training...")
corpus_sentences = prepare_corpus_for_embeddings(train_subset)
print(f"Total sentences: {len(corpus_sentences)}")

# Train Word2Vec model (lightweight approach)
EMBEDDING_DIM = 50  # Smaller dimensions for efficiency
print(f"\nTraining Word2Vec model with {EMBEDDING_DIM} dimensions...")

w2v_model = Word2Vec(
    sentences=corpus_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    workers=4,
    epochs=5,  # Reduced epochs for speed
    sg=0  # CBOW algorithm (faster than Skip-gram)
)

print(f"Word2Vec model trained successfully!")
print(f"Vocabulary size: {len(w2v_model.wv.key_to_index)}")

# Test semantic similarity
try:
    test_words = ['tốt', 'ngon', 'đẹp', 'chất lượng']
    for word in test_words:
        if word in w2v_model.wv:
            similar = w2v_model.wv.most_similar(word, topn=3)
            print(f"Words similar to '{word}': {[w for w, s in similar]}")
            break
except:
    print("Testing similarity (some words might not be in vocabulary)")

print("\nWord2Vec embeddings ready for semantic scoring!")

Preparing corpus for Word2Vec training...
Total sentences: 6000

Training Word2Vec model with 50 dimensions...
Word2Vec model trained successfully!
Vocabulary size: 5258
Words similar to 'tốt': ['giá.', 'ok.', 'giá,']

Word2Vec embeddings ready for semantic scoring!


## Step 5: Token-level F1 Evaluation Metric

Implement the official evaluation metric as specified in competition guidelines.

In [6]:
def normalize_text(text):
    """Normalize text for token-level evaluation"""
    if pd.isna(text) or text is None:
        return ""
    
    # Convert to lowercase and remove extra whitespace
    text = str(text).lower().strip()
    
    # Remove punctuation and extra spaces
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def tokenize_text(text):
    """Tokenize text by splitting on whitespace"""
    normalized = normalize_text(text)
    if not normalized:
        return []
    return normalized.split()

def calculate_token_overlap(pred_tokens, true_tokens):
    """Calculate token overlap count between predicted and true tokens"""
    pred_count = Counter(pred_tokens)
    true_count = Counter(true_tokens)
    
    overlap = 0
    for token in pred_count:
        if token in true_count:
            overlap += min(pred_count[token], true_count[token])
    
    return overlap

def calculate_token_f1(y_pred, y_true):
    """
    Calculate Token-level F1 score according to competition specification
    
    For each sample i:
    - Tokenize and normalize predicted and true answers: T̂_i and T_i
    - Calculate overlap_i = sum over tokens t of min(count(t,T̂_i), count(t,T_i))
    - Precision_i = overlap_i / |T̂_i| if |T̂_i| > 0, else 0
    - Recall_i = overlap_i / |T_i| if |T_i| > 0, else 0  
    - F1_i = 2 * Precision_i * Recall_i / (Precision_i + Recall_i) if Precision_i + Recall_i > 0, else 0
    - Final F1 = macro average = (1/N) * sum(F1_i)
    """
    assert len(y_pred) == len(y_true), "Prediction and ground truth lists must have same length"
    
    f1_scores = []
    precision_scores = []
    recall_scores = []
    
    for pred, true in zip(y_pred, y_true):
        # Tokenize predicted and true answers
        pred_tokens = tokenize_text(pred)
        true_tokens = tokenize_text(true)
        
        # Calculate overlap
        overlap = calculate_token_overlap(pred_tokens, true_tokens)
        
        # Calculate precision
        if len(pred_tokens) > 0:
            precision = overlap / len(pred_tokens)
        else:
            precision = 0.0
            
        # Calculate recall  
        if len(true_tokens) > 0:
            recall = overlap / len(true_tokens)
        else:
            recall = 0.0
            
        # Calculate F1
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0
            
        f1_scores.append(f1)
        precision_scores.append(precision)
        recall_scores.append(recall)
    
    # Return macro averages
    macro_f1 = np.mean(f1_scores)
    macro_precision = np.mean(precision_scores)
    macro_recall = np.mean(recall_scores)
    
    return {
        'token_level_f1': macro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'individual_f1': f1_scores
    }

# Test the evaluation function
print("Testing Token-level F1 evaluation function...")

# Test cases for Vietnamese
test_pred = ["xe đẹp lắm", "chất lượng tốt", ""]
test_true = ["xe rất đẹp", "chất lượng", "không có gì"]

test_results = calculate_token_f1(test_pred, test_true)
print(f"Test results:")
print(f"Token-level F1: {test_results['token_level_f1']:.4f}")
print(f"Macro Precision: {test_results['macro_precision']:.4f}")
print(f"Macro Recall: {test_results['macro_recall']:.4f}")

print("\nOfficial Token-level F1 evaluation function implemented successfully!")

Testing Token-level F1 evaluation function...
Test results:
Token-level F1: 0.4889
Macro Precision: 0.4444
Macro Recall: 0.5556

Official Token-level F1 evaluation function implemented successfully!


## Step 6: Baseline Model Implementation

Hybrid model combining TF-IDF, Word2Vec embeddings, and ensemble scoring.

In [7]:
class ViQABaseline:
    """Baseline combining TF-IDF + Word2Vec + ensemble scoring"""
    
    def __init__(self, w2v_model=None):
        self.tfidf_vectorizer = TfidfVectorizer(
            ngram_range=(1, 3),  # Unigrams to trigrams
            max_features=10000,  # Limit vocabulary size
            lowercase=True,
            stop_words=None  # Keep all words for Vietnamese
        )
        self.w2v_model = w2v_model
        self.is_fitted = False
        
    def fit(self, reviews, questions, answers):
        """Fit the model on training data"""
        # Prepare corpus for TF-IDF
        corpus = []
        for review, question, answer in zip(reviews, questions, answers):
            # Combine all text for better TF-IDF representation
            combined_text = f"{clean_vietnamese_text(review)} {clean_vietnamese_text(question)} {clean_vietnamese_text(answer)}"
            corpus.append(combined_text)
        
        # Fit TF-IDF
        self.tfidf_vectorizer.fit(corpus)
        self.is_fitted = True
        
        print(f" baseline fitted on {len(corpus)} samples")
        print(f"TF-IDF vocabulary size: {len(self.tfidf_vectorizer.vocabulary_)}")
        
    def calculate_semantic_similarity(self, text1, text2):
        """Calculate semantic similarity using Word2Vec embeddings"""
        if not self.w2v_model:
            return 0.0
        
        words1 = clean_vietnamese_text(text1).split()
        words2 = clean_vietnamese_text(text2).split()
        
        # Get word vectors that exist in vocabulary
        vecs1 = [self.w2v_model.wv[word] for word in words1 if word in self.w2v_model.wv]
        vecs2 = [self.w2v_model.wv[word] for word in words2 if word in self.w2v_model.wv]
        
        if not vecs1 or not vecs2:
            return 0.0
        
        # Average word vectors to get text representation
        vec1_avg = np.mean(vecs1, axis=0)
        vec2_avg = np.mean(vecs2, axis=0)
        
        # Calculate cosine similarity
        dot_product = np.dot(vec1_avg, vec2_avg)
        norm1 = np.linalg.norm(vec1_avg)
        norm2 = np.linalg.norm(vec2_avg)
        
        if norm1 == 0 or norm2 == 0:
            return 0.0
        
        return dot_product / (norm1 * norm2)
    
    def calculate_tfidf_similarity(self, question, candidate):
        """Calculate TF-IDF based similarity"""
        try:
            # Transform texts to TF-IDF vectors
            question_vec = self.tfidf_vectorizer.transform([clean_vietnamese_text(question)])
            candidate_vec = self.tfidf_vectorizer.transform([clean_vietnamese_text(candidate)])
            
            # Calculate cosine similarity
            similarity = cosine_similarity(question_vec, candidate_vec)[0][0]
            return similarity
        except:
            return 0.0
    
    def calculate_ngram_overlap(self, question, candidate):
        """Calculate n-gram overlap score"""
        # Character-level n-grams
        char_ngrams_q = set(extract_ngrams(question, 3))
        char_ngrams_c = set(extract_ngrams(candidate, 3))
        
        char_overlap = len(char_ngrams_q & char_ngrams_c) / max(len(char_ngrams_q | char_ngrams_c), 1)
        
        # Word-level n-grams  
        word_ngrams_q = set(extract_word_ngrams(question, 2))
        word_ngrams_c = set(extract_word_ngrams(candidate, 2))
        
        word_overlap = len(word_ngrams_q & word_ngrams_c) / max(len(word_ngrams_q | word_ngrams_c), 1)
        
        return (char_overlap + word_overlap) / 2
    
    def extract_candidates(self, review, window_size=20, step=5):
        """Extract candidate answer spans from review"""
        review_clean = clean_vietnamese_text(review)
        words = review_clean.split()
        
        candidates = []
        
        # Extract sliding windows of different sizes
        for w_size in range(1, min(window_size + 1, len(words) + 1)):
            for i in range(0, len(words) - w_size + 1, step):
                candidate = ' '.join(words[i:i + w_size])
                candidates.append(candidate)
        
        # Add single words as candidates
        candidates.extend(words)
        
        # Remove duplicates
        return list(set(candidates))
    
    def find_answer(self, review, question):
        """Find best answer using ensemble scoring"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")
        
        # Extract candidate spans
        candidates = self.extract_candidates(review)
        
        if not candidates:
            return ""
        
        best_answer = ""
        best_score = -1
        
        # Score each candidate using ensemble approach
        for candidate in candidates:
            # 1. TF-IDF similarity (weight: 0.4)
            tfidf_sim = self.calculate_tfidf_similarity(question, candidate)
            
            # 2. Semantic similarity via Word2Vec (weight: 0.3)
            semantic_sim = self.calculate_semantic_similarity(question, candidate)
            
            # 3. N-gram overlap (weight: 0.3)
            ngram_sim = self.calculate_ngram_overlap(question, candidate)
            
            # Ensemble score with weights
            ensemble_score = (0.4 * tfidf_sim + 
                            0.3 * semantic_sim + 
                            0.3 * ngram_sim)
            
            # Bonus for answer length (prefer reasonable length answers)
            length_bonus = min(len(candidate.split()), 5) / 5.0 * 0.1
            final_score = ensemble_score + length_bonus
            
            if final_score > best_score:
                best_score = final_score
                best_answer = candidate
        
        return best_answer.strip()

print("ViQA Baseline model class implemented successfully!")
print("Features: TF-IDF + Word2Vec + N-gram matching + Ensemble scoring")

ViQA Baseline model class implemented successfully!
Features: TF-IDF + Word2Vec + N-gram matching + Ensemble scoring


## Step 7: Train Baseline Model

Fit the model on training data.

In [8]:
# Initialize and train model
print("Initializing ViQA Baseline...")
model = ViQABaseline(w2v_model=w2v_model)

# Prepare training data
train_reviews = train_subset['review'].tolist()
train_questions = train_subset['question'].tolist() 
train_answers = train_subset['answer'].tolist()

print("Training  baseline model...")
model.fit(train_reviews, train_questions, train_answers)

print("\nModel training completed!")

# Test the model with a sample
print("\nTesting model with sample data:")
sample_idx = 0
sample_review = train_subset.iloc[sample_idx]['review']
sample_question = train_subset.iloc[sample_idx]['question']
sample_true_answer = train_subset.iloc[sample_idx]['answer']

predicted_answer = model.find_answer(sample_review, sample_question)

print(f"Question: {sample_question}")
print(f"True Answer: {sample_true_answer}")
print(f"Predicted Answer: {predicted_answer}")

# Calculate Token-level F1 for this sample
sample_f1 = calculate_token_f1([predicted_answer], [sample_true_answer])
print(f"Sample Token-level F1: {sample_f1['token_level_f1']:.4f}")

print("\n baseline model ready for evaluation!")

Initializing ViQA Baseline...
Training  baseline model...
 baseline fitted on 2000 samples
TF-IDF vocabulary size: 10000

Model training completed!

Testing model with sample data:
Question: Quán phù hợp nhất cho những khách hàng có mục đích gì?
True Answer: thích hợp cho việc hộp nhóm , đọc sách , nghe nhạc , thư giãn
Predicted Answer: nơi đây thích hợp cho việc
Sample Token-level F1: 0.4444

 baseline model ready for evaluation!


## Step 8: Validation Set Evaluation

Comprehensive evaluation using the official Token-level F1 metric.

In [9]:
# Evaluate on validation set
print("="*60)
print(" BASELINE VALIDATION EVALUATION")
print("="*60)

# Generate predictions on validation set
print("Generating predictions on validation set...")
val_predictions = []
val_true = val_subset['answer'].tolist()

for idx, row in val_subset.iterrows():
    predicted = model.find_answer(row['review'], row['question'])
    val_predictions.append(predicted)
    
    if (len(val_predictions)) % 50 == 0:
        print(f"Processed {len(val_predictions)}/{len(val_subset)} samples")

print(f"Generated {len(val_predictions)} predictions")

# Calculate Token-level F1 results
eval_results = calculate_token_f1(val_predictions, val_true)

print(f"\n BASELINE RESULTS:")
print(f"Token-level F1: {eval_results['token_level_f1']:.4f}")
print(f"Macro Precision: {eval_results['macro_precision']:.4f}")
print(f"Macro Recall: {eval_results['macro_recall']:.4f}")

 BASELINE VALIDATION EVALUATION
Generating predictions on validation set...
Processed 50/300 samples
Processed 100/300 samples
Processed 150/300 samples
Processed 200/300 samples
Processed 250/300 samples
Processed 300/300 samples
Generated 300 predictions

 BASELINE RESULTS:
Token-level F1: 0.1412
Macro Precision: 0.1526
Macro Recall: 0.2094


## Step 9: Generate Test Predictions

Generate predictions for test set and create submission file.

In [10]:
# Generate predictions for test set
print("Generating predictions for test set...")
test_predictions = []

for idx, row in test_df.iterrows():
    predicted = model.find_answer(row['review'], row['question'])
    test_predictions.append(predicted)
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} samples")

print(f"Generated predictions for {len(test_predictions)} test samples")

# Create submission dataframe
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'answer': test_predictions
})

print(f"\nSubmission format:")
print(f"Shape: {submission_df.shape}")
print(f"Columns: {submission_df.columns.tolist()}")
print("\nFirst 5 predictions:")
print(submission_df.head())

# Check for empty predictions
empty_test_predictions = submission_df['answer'].isna().sum() + sum(1 for ans in test_predictions if not str(ans).strip())
print(f"\nEmpty test predictions: {empty_test_predictions}/{len(test_predictions)}")

Generating predictions for test set...
Processed 100/890 samples
Processed 200/890 samples
Processed 300/890 samples
Processed 400/890 samples
Processed 500/890 samples
Processed 600/890 samples
Processed 700/890 samples
Processed 800/890 samples
Generated predictions for 890 test samples

Submission format:
Shape: (890, 2)
Columns: ['id', 'answer']

First 5 predictions:
   id                                             answer
0   0            google assistant. camera cho chất lượng
1   1                         hội chị em . lúc ship hàng
2   2  dc 1,5 bạn nào ko mê game như mình thì ko nên ...
3   3                         trong cái bảng điều khiển,
4   4                             đánh giá một sao nhiều

Empty test predictions: 0/890


## Step 10: Save Submission and Performance Summary

Save predictions and summarize model performance.

In [11]:
# Save submission file
submission_filename = "Baseline_ViQA_Task3.csv"
submission_df.to_csv(submission_filename, index=False)
print(f"Submission file saved as: {submission_filename}")

# Verify submission file
print(f"\nSubmission file verification:")
try:
    verify_df = pd.read_csv(submission_filename)
    print(f" File saved successfully")
    print(f"Shape: {verify_df.shape}")
    print(f"Columns: {verify_df.columns.tolist()}")
    print(f"Sample rows:")
    print(verify_df.head(3))
except Exception as e:
    print(f"Error reading submission file: {e}")

Submission file saved as: Baseline_ViQA_Task3.csv

Submission file verification:
 File saved successfully
Shape: (890, 2)
Columns: ['id', 'answer']
Sample rows:
   id                                             answer
0   0            google assistant. camera cho chất lượng
1   1                         hội chị em . lúc ship hàng
2   2  dc 1,5 bạn nào ko mê game như mình thì ko nên ...
